# Agentspace + AlloyDB AI NL2SQL

This notebook provides an example of setting up AlloyDB AI NL2SQL as a Data Source for search within Agentspace.

References:
- [Generate SQL Queries with AlloyDB AI NL2SQL](https://cloud.google.com/alloydb/docs/ai/generate-sql-queries-natural-language)
- [Setup AlloyDB AI NL2SQL in Agentspace](https://cloud.google.com/agentspace/agentspace-enterprise/docs/create-data-store#alloydb-ai-nl-setup)

> NOTE: Both Agentspace and AlloyDB AI NL2SQL are in Preview. Please work with your Google account team to enable the features in your project before running the steps in this notebook.


## Basic Setup

You will need an AlloyDB for PostgreSQL instance to use this notebook. Create one now if you have not already created it.

### Define Variables

In [ ]:
# Update these variables to match your environment
project_id = "your-project"  # @param {type:"string"}
region = "your-region"  # @param {type:"string"}
vpc = "your-vpc"  # @param {type:"string"}
alloydb_cluster = "your-alloydb-cluster"  # @param {type:"string"}
alloydb_instance = "your-alloydb-instance"  # @param {type:"string"}
alloydb_database = "ecom" # @param {type:"string"}
alloydb_password = input("Please provide a password to be used for 'postgres' database user: ")
agentspace_user_password = input("Please provide a password to be used for 'agentspace_user' database user: ")


### Install Dependencies

In [ ]:
! pip install --quiet google-cloud-storage \
                      google-cloud-aiplatform \
                      asyncpg \
                      google.cloud.alloydb.connector

### Connect Your Google Cloud Project

In [ ]:
# Configure gcloud.
!gcloud config set project {project_id}

### Configure Logging

In [ ]:
import logging
import sys

# Configure the root logger to output messages with INFO level or above
logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s[%(levelname)5s][%(name)14s] - %(message)s',  datefmt='%H:%M:%S', force=True)

### Enable APIs for AlloyDB, Vertex AI, and Discovery Engine

You will need to enable these APIs in order to create an AlloyDB database and utilize Vertex AI as an embeddings service!

In [ ]:
!gcloud services enable alloydb.googleapis.com aiplatform.googleapis.com discoveryengine.googleapis.com

### Initialize GenAI Client

In [ ]:
from google import genai
from google.genai import types

genai_client = genai.Client(
    vertexai=True, project=project_id, location=region
)

### Connect to the AlloyDB Cluster

This function will create a connection pool to your AlloyDB instance using the AlloyDB Python connector. The AlloyDB Python connector will automatically create secure connections to your AlloyDB instance using mTLS.

In [ ]:
import asyncpg

import sqlalchemy
from sqlalchemy.ext.asyncio import AsyncEngine, create_async_engine

from google.cloud.alloydb.connector import AsyncConnector, IPTypes

async def init_connection_pool(connector: AsyncConnector, db_name: str = alloydb_database, pool_size: int = 5) -> AsyncEngine:
    # initialize Connector object for connections to AlloyDB
    connection_string = f"projects/{project_id}/locations/{region}/clusters/{alloydb_cluster}/instances/{alloydb_instance}"

    async def getconn() -> asyncpg.Connection:
        conn: asyncpg.Connection = await connector.connect(
            connection_string,
            "asyncpg",
            user="postgres",
            password=alloydb_password,
            db=db_name,
            ip_type=IPTypes.PRIVATE,
        )
        return conn

    pool = create_async_engine(
        "postgresql+asyncpg://",
        async_creator=getconn,
        pool_size=pool_size,
        max_overflow=0,
        isolation_level='AUTOCOMMIT'
    )
    return pool

connector = AsyncConnector()

postgres_db_pool = await init_connection_pool(connector, "postgres")
ecom_db_pool = await init_connection_pool(connector, f"{alloydb_database}")

## Define Helper Functions

#### rest_api_helper()

In [ ]:
import requests
import google.auth
import json

# Get an access token based upon the current user
creds, _ = google.auth.default()
authed_session = google.auth.transport.requests.AuthorizedSession(creds)
access_token=creds.token

if project_id:
  authed_session.headers.update({"X-Goog-User-Project": project_id}) # Required to workaround a project quota bug

def rest_api_helper(
    session: requests.Session,
    url: str,
    http_verb: str,
    request_body: dict = None,
    params: dict = None
  ) -> dict:
  """Calls a REST API using a pre-authenticated requests Session."""

  headers = {"Content-Type": "application/json"}

  try:

    if http_verb == "GET":
      response = session.get(url, headers=headers, params=params)
    elif http_verb == "POST":
      response = session.post(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PUT":
      response = session.put(url, json=request_body, headers=headers, params=params)
    elif http_verb == "PATCH":
      response = session.patch(url, json=request_body, headers=headers, params=params)
    elif http_verb == "DELETE":
      response = session.delete(url, headers=headers, params=params)
    else:
      raise ValueError(f"Unknown HTTP verb: {http_verb}")

    # Raise an exception for bad status codes (4xx or 5xx)
    response.raise_for_status()

    # Check if response has content before trying to parse JSON
    if response.content:
        return response.json()
    else:
        return {} # Return empty dict for empty responses (like 204 No Content)

  except requests.exceptions.RequestException as e:
      # Catch potential requests library errors (network, timeout, etc.)
      # Log detailed error information
      print(f"Request failed: {e}")
      if e.response is not None:
          print(f"Request URL: {e.request.url}")
          print(f"Request Headers: {e.request.headers}")
          print(f"Request Body: {e.request.body}")
          print(f"Response Status: {e.response.status_code}")
          print(f"Response Text: {e.response.text}")
          # Re-raise a more specific error or a custom one
          raise RuntimeError(f"API call failed with status {e.response.status_code}: {e.response.text}") from e
      else:
          raise RuntimeError(f"API call failed: {e}") from e
  except json.JSONDecodeError as e:
      print(f"Failed to decode JSON response: {e}")
      print(f"Response Text: {response.text}")
      raise RuntimeError(f"Invalid JSON received from API: {response.text}") from e



#### run_query()

In [ ]:
# Create AlloyDB Query Helper Function
import sqlalchemy
from sqlalchemy import text, exc
import pandas as pd

async def run_query(pool, sql: str, params = None, output_as_df: bool = True):
    """Executes a SQL query or statement against the database pool.

    Handles various SQL statements:
    - SELECT/WITH: Returns results as a DataFrame (if output_as_df=True)
      or ResultProxy. Supports parameters. Does not commit.
    - EXPLAIN/EXPLAIN ANALYZE: Executes the explain, returns the query plan
      as a formatted multi-line string. Ignores output_as_df.
      Supports parameters. Does not commit.
    - INSERT/UPDATE/DELETE/CREATE/ALTER etc.: Executes the statement,
      commits the transaction, logs info, and returns the ResultProxy.
      Supports single or bulk parameters (executemany).

    Args:
      pool: An asynchronous SQLAlchemy connection pool.
      sql: A string containing the SQL query or statement template.
      params: Optional.
        - None: Execute raw SQL (Use with caution for non-SELECT/EXPLAIN).
        - dict or tuple: Parameters for a single execution.
        - list of dicts/tuples: Parameters for bulk execution (executemany).
      output_as_df (bool): If True and query is SELECT/WITH, return pandas DataFrame.
                           Ignored for EXPLAIN and non-data-returning statements.

    Returns:
      pandas.DataFrame | str | sqlalchemy.engine.Result | None:
        - DataFrame: For SELECT/WITH if output_as_df=True.
        - str: For EXPLAIN/EXPLAIN ANALYZE, containing the formatted query plan.
        - ResultProxy: For non-SELECT/WITH/EXPLAIN statements, or SELECT/WITH
                       if output_as_df=False.
        - None: If a SQLAlchemy ProgrammingError or other specific error occurs.

    Raises:
        Exception: Catches and logs `sqlalchemy.exc.ProgrammingError`, returning None.
                   May re-raise other database exceptions.

    Example Execution:
      Single SELECT:
        sql_select = "SELECT ticker, company_name from investments LIMIT 5"
        df_result = await run_query(pool, sql_select)

      Single non-SELECT - Parameterized (Safe!):
        Parameterized INSERT:
          sql_insert = "INSERT INTO investments (ticker, company_name) VALUES (:ticker, :name)"
          params_insert = {"ticker": "NEW", "name": "New Company"}
          insert_result = await run_query(pool, sql_insert, params_insert)

        Parameterized UPDATE:
          sql_update = "UPDATE products SET price = :price WHERE id = :product_id"
          params_update = {"price": 99.99, "product_id": 123}
          update_result = await run_query(pool, sql_update, params_update)

      Bulk Update:
        docs = pd.DataFrame([
            {'id': 101, 'sparse_embedding': '[0.1, 0.2]'},
            {'id': 102, 'sparse_embedding': '[0.3, 0.4]'},
            # ... more rows
        ])

        update_sql_template = '''
            UPDATE products
            SET sparse_embedding = :embedding,
                sparse_embedding_model = 'BM25'
            WHERE id = :product_id
        ''' # Using named parameters :param_name

        # Prepare list of dictionaries for params
        data_to_update = [
            {"embedding": row.sparse_embedding, "product_id": row.id}
            for row in docs.itertuples(index=False)
        ]

        if data_to_update:
          bulk_result = await run_query(pool, update_sql_template, data_to_update)
          # bulk_result is the SQLAlchemy ResultProxy

    """
    sql_lower_stripped = sql.strip().lower()
    is_select_with = sql_lower_stripped.startswith(('select', 'with'))
    is_explain = sql_lower_stripped.startswith('explain')

    # Determine if the statement is expected to return data rows or a plan
    is_data_returning = is_select_with or is_explain

    # Determine actual DataFrame output eligibility (only for SELECT/WITH)
    effective_output_as_df = output_as_df and is_select_with

    # Check if params suggest a bulk operation (for logging purposes)
    is_bulk_operation = isinstance(params, (list, tuple)) and len(params) > 0 and isinstance(params[0], (dict, tuple, list))

    async with pool.connect() as conn:
        try:
          # Execute with or without params
          if params:
              result = await conn.execute(text(sql), params)
          else:
              # Add warning for raw SQL only if it's NOT data-returning
              #if not is_data_returning:
                  #logging.warning("Executing non-SELECT/EXPLAIN raw SQL without parameters. Ensure SQL is safe.")
              result = await conn.execute(text(sql))

          # --- Handle statements that return data or plan ---
          if is_data_returning:
              if is_explain:
                  # Fetch and format EXPLAIN output as a string
                    try:
                        plan_rows = result.fetchall()
                        # EXPLAIN output is usually text in the first column
                        query_plan = "\n".join([str(row[0]) for row in plan_rows])
                        #logging.info(f"EXPLAIN executed successfully for: {sql[:100]}...")
                        return query_plan
                    except Exception as e:
                        logging.error(f"Error fetching/formatting EXPLAIN result: {e}")
                        return None
              else: # Handle SELECT / WITH
                  if effective_output_as_df:
                      try:
                          rows = result.fetchall()
                          column_names = result.keys()
                          df = pd.DataFrame(rows, columns=column_names)
                          #logging.info(f"SELECT/WITH executed successfully, returning DataFrame for: {sql[:100]}...")
                          return df
                      except Exception as e:
                          logging.error(f"Error converting SELECT result to DataFrame: {e}")
                          logging.info(f"Returning raw ResultProxy for SELECT/WITH due to DataFrame conversion error for: {sql[:100]}...")
                          return result # Fallback to raw result
                  else:
                      # Return raw result proxy for SELECT/WITH if df output not requested
                      #logging.info(f"SELECT/WITH executed successfully, returning ResultProxy for: {sql[:100]}...")
                      return result

          # --- Handle Non-Data Returning Statements (INSERT, UPDATE, DELETE, CREATE, etc.) ---
          else:
              await conn.commit() # Commit changes ONLY for these statements
              operation_type = sql.strip().split()[0].upper()
              row_count = result.rowcount # Note: rowcount behavior varies

              if is_bulk_operation:
                  print(f"Bulk {operation_type} executed for {len(params)} items. Result rowcount: {row_count}")
              elif operation_type in ['INSERT', 'UPDATE', 'DELETE']:
                  print(f"{operation_type} statement executed successfully. {row_count} row(s) affected.")
              else: # CREATE, ALTER, etc.
                  print(f"{operation_type} statement executed successfully. Result rowcount: {row_count}")
              return result # Return the result proxy

        except exc.ProgrammingError as e:
            # Log the error with context
            logging.error(f"SQL Programming Error executing query:\nSQL: {sql[:500]}...\nParams (sample): {str(params)[:500]}...\nError: {e}")
            # Rollback might happen automatically on context exit with error, but explicit can be clearer
            # await conn.rollback() # Consider if needed based on pool/transaction settings
            return None # Return None on handled programming errors
        except Exception as e:
            # Log other unexpected errors
            logging.error(f"An unexpected error occurred during query execution:\nSQL: {sql[:500]}...\nError: {e}")
            # await conn.rollback() # Consider if needed
            raise # Re-raise unexpected errors



## Setup AlloyDB NL2SQL

Reference: https://cloud.google.com/alloydb/docs/ai/generate-sql-queries-natural-language

### Enable the extension

In [ ]:
sql_array = []

#sql_array.append("CREATE EXTENSION google_ml_integration with version '1.4.2';")
sql_array.append("CREATE EXTENSION alloydb_ai_nl cascade;")

for sql in sql_array:
  await run_query(ecom_db_pool, sql)

### Register the Gemini 2.0 Flash Model

In [ ]:
sql = f"""CALL google_ml.create_model(
    model_id => 'gemini-2_0_flash',
    model_request_url => 'https://{region}-aiplatform.googleapis.com/v1/projects/{project_id}/locations/{region}/publishers/google/models/gemini-2.0-flash:streamGenerateContent',
    model_provider => 'google',
    model_auth_type => 'alloydb_service_agent_iam');"""

await run_query(ecom_db_pool, sql)

### Setup a Default nl_config

In [ ]:
sql = """SELECT alloydb_ai_nl.g_manage_configuration(
'create_configuration', -- operation
'default', -- configuration_id_in
'gemini-2_0_flash' -- model_id_in
);"""

await run_query(ecom_db_pool, sql)

### Add Some Examples

In [ ]:
sql_array = []

sql_array.append("""SELECT alloydb_ai_nl.add_example(
	nl_example => 'Run a vector search on the products table that searches the embedding column for the string "coach purse".',
	sql_example => 'SELECT
      products.embedding <=> embedding (''text-embedding-005'', ''Coach purse'')::vector AS distance,
      products.name,
      products.product_image_uri,
      products.brand,
      products.product_description,
      products.category,
      products.department,
      products.cost,
      products.retail_price::MONEY,
      products.sku,
      ''VECTOR'' AS retrieval_method
    FROM
      public.products
    ORDER BY
      distance
    LIMIT
      12',
  context_example => 'default',
	explanation_example => 'This SQL query uses the AlloyDB AI embedding() function to get an embedding for the search phrase "coach purse", then it compares the generated embedding to vector embeddings stored in the embedding column. It returns the 12 most relevant results based on vector distances of the search embedding and the stored embeddings. We use the products.embedding field because the question is about products.'
)""")

sql_array.append("""SELECT alloydb_ai_nl.add_example(
	nl_example => 'Show me all items that are out of stock.',
	sql_example => 'SELECT p.id, p.name
      FROM products p
      LEFT JOIN inventory_items i ON p.id = i.product_id AND i.sold_at IS NULL
      WHERE i.id IS NULL',
  context_example => 'default',
	explanation_example => 'In this context, items and products mean the same thing.');
""")

sql_array.append("""SELECT alloydb_ai_nl.add_example(
	nl_example => 'How many luxury women''s items do we have in stock?',
	sql_example => 'WITH on_hand AS (
  SELECT product_id, product_name, COUNT(*) AS on_hand_count FROM inventory_items i
  WHERE i.sold_at IS NULL
  GROUP BY i.product_id, i.product_name
  HAVING COUNT(id) > 0
)
SELECT
  products.embedding <=> embedding (''text-embedding-005'', ''luxury womens items'')::vector AS distance,
  products.id,
  products.name,
  products.brand,
  products.department,
  products.cost,
  products.retail_price,
  products.sku,
  on_hand.on_hand_count
FROM products
JOIN on_hand ON products.id = on_hand.product_id
ORDER BY distance DESC
LIMIT 10;',
  context_example => 'default',
	explanation_example => 'We calculate on-hand counts so that we only show in-stock items');
""")

sql_array.append("""SELECT alloydb_ai_nl.add_example(
	nl_example => 'What were my total sales yesterday? Include sales for the last day, week, month, and year.',
	sql_example => 'WITH RelevantLatestDate AS (
  -- Find the date of the most recent relevant order item.
  -- Filter statuses here to ensure the reference date is based on actual sales data.
  -- Use COALESCE to provide a default (e.g., today) if the table has no relevant data,
  -- although the outer query''s COALESCE(SUM...) might handle a NULL date propagating.
  -- Let''s assume we want results relative to *some* date even if no orders exist.
  -- If you prefer NULL/no results when empty, remove the COALESCE here.
  SELECT
     COALESCE(MAX(oi.created_at)::DATE, CURRENT_DATE) AS latest_date
  FROM order_items oi
  WHERE oi.status NOT IN (''Cancelled'', ''Returned'')

), TimeBoundaries AS (
  -- Calculate time boundaries relative to the latest date found in the data.
  SELECT
    -- The reference date (start of the day containing the latest data)
    rld.latest_date, -- Keep the date itself for clarity if needed
    DATE_TRUNC(''day'', rld.latest_date)::TIMESTAMP AS start_of_latest_data_day, -- e.g., 2024-10-15 00:00:00 if max(created_at) was on Oct 15th

    -- Day *before* the latest data day
    DATE_TRUNC(''day'', rld.latest_date - INTERVAL ''1 day'')::TIMESTAMP AS start_of_day_before_latest, -- e.g., 2024-10-14 00:00:00

    -- Start of the 7-day period *ending* at the beginning of the latest data day
    DATE_TRUNC(''day'', rld.latest_date - INTERVAL ''7 day'')::TIMESTAMP AS start_of_7_days_before_latest, -- e.g., 2024-10-08 00:00:00

    -- Last Calendar Month boundaries relative to the latest data day
    DATE_TRUNC(''month'', rld.latest_date - INTERVAL ''1 month'')::TIMESTAMP AS start_of_prev_calendar_month_relative, -- e.g., 2024-09-01 00:00:00
    DATE_TRUNC(''month'', rld.latest_date)::TIMESTAMP AS start_of_latest_data_calendar_month, -- e.g., 2024-10-01 00:00:00 (Exclusive end boundary for previous month)

    -- Last Calendar Year boundaries relative to the latest data day
    DATE_TRUNC(''year'', rld.latest_date - INTERVAL ''1 year'')::TIMESTAMP AS start_of_prev_calendar_year_relative, -- e.g., 2023-01-01 00:00:00
    DATE_TRUNC(''year'', rld.latest_date)::TIMESTAMP AS start_of_latest_data_calendar_year -- e.g., 2024-01-01 00:00:00 (Exclusive end boundary for previous year)

  FROM RelevantLatestDate rld -- Use the dynamically found latest date

) SELECT
  -- Renamed aliases for clarity, reflecting they are relative to the latest data day

  -- Sales for the Day Before the Latest Data Day
  COALESCE(SUM(CASE WHEN oi.created_at >= tb.start_of_day_before_latest AND oi.created_at < tb.start_of_latest_data_day THEN oi.sale_price ELSE 0 END), 0) AS sales_day_before_latest,

  -- Sales for 7 Days ending at the start of the Latest Data Day
  COALESCE(SUM(CASE WHEN oi.created_at >= tb.start_of_7_days_before_latest AND oi.created_at < tb.start_of_latest_data_day THEN oi.sale_price ELSE 0 END), 0) AS sales_7_days_before_latest,

  -- Sales for the Calendar Month Before the Month Containing the Latest Data
  COALESCE(SUM(CASE WHEN oi.created_at >= tb.start_of_prev_calendar_month_relative AND oi.created_at < tb.start_of_latest_data_calendar_month THEN oi.sale_price ELSE 0 END), 0) AS sales_prev_calendar_month_relative,

  -- Sales for the Calendar Year Before the Year Containing the Latest Data
  COALESCE(SUM(CASE WHEN oi.created_at >= tb.start_of_prev_calendar_year_relative AND oi.created_at < tb.start_of_latest_data_calendar_year THEN oi.sale_price ELSE 0 END), 0) AS sales_prev_calendar_year_relative

FROM
  order_items AS oi
CROSS JOIN -- Join with the single row of calculated time boundaries
  TimeBoundaries AS tb
WHERE
  -- Filter statuses *again* here for the actual aggregation
  oi.status NOT IN (''Cancelled'', ''Returned'')

  -- Pre-filter rows based on the widest *dynamic* time range needed
  -- Only consider items from the start of the earliest period (previous calendar year relative to latest data)
  -- up to the *start* of the latest data day (exclusive).
  AND oi.created_at >= tb.start_of_prev_calendar_year_relative
  AND oi.created_at < tb.start_of_latest_data_day; -- Important: Ensure upper bound aligns with periods being measured',
  context_example => 'default',
	explanation_example => 'This example gets the most recent total sales numbers for the last day, week, month, and year. The database may not be up to date, so we consider the most recent day of data in the database as "yesterday".');
""")

for sql in sql_array:
  await run_query(ecom_db_pool, sql)

### View Examples

In [ ]:
sql = "SELECT * FROM alloydb_ai_nl.example_store_view;"
await run_query(ecom_db_pool, sql)

### Test the NL2SQL Configuration

In [ ]:
sql = """SELECT alloydb_ai_nl.get_sql_agentic(
      'default',  --nl_config
      'What were my total sales yesterday?' -- nl question
);"""
generated_sql = await run_query(ecom_db_pool, sql)
generated_sql = generated_sql.iloc[0, 0]
#print(f"Generated SQL: {generated_sql}")

await run_query(ecom_db_pool, generated_sql)

## Setup Agentspace

### Create Agentspace User in AlloyDB

In [ ]:
sql_array = []

sql_array.append(f"CREATE ROLE agentspace_user WITH LOGIN PASSWORD '{agentspace_user_password}';")
sql_array.append("GRANT SELECT ON TABLE public.products TO agentspace_user;")
sql_array.append("GRANT SELECT ON TABLE public.orders TO agentspace_user;")
sql_array.append("GRANT SELECT ON TABLE public.order_items TO agentspace_user;")
sql_array.append("GRANT SELECT ON TABLE public.inventory_items TO agentspace_user;")
sql_array.append("GRANT SELECT ON TABLE public.users TO agentspace_user;")
sql_array.append("GRANT SELECT ON TABLE public.events TO agentspace_user;")
sql_array.append("GRANT SELECT ON TABLE public.distribution_centers TO agentspace_user;")

for sql in sql_array:
  await run_query(ecom_db_pool, sql)


### Create an AlloyDB Agentspace Data Store

In [ ]:
data_store_id = "alloydb-ecom-2"

url = f"https://discoveryengine.googleapis.com/v1alpha/projects/{project_id}/locations/global/collections/default_collection/dataStores?dataStoreId={data_store_id}"
request_body = {
      "displayName": "AlloyDB ecom Database",
      "industryVertical": "GENERIC",
      "solutionTypes": ["SOLUTION_TYPE_SEARCH"],
      "federatedSearchConfig": {
        "alloyDbConfig": {
          "alloydbConnectionConfig": {
            "instance": f"projects/{project_id}/locations/{region}/clusters/{alloydb_cluster}/instances/{alloydb_instance}",
            "database": f"{alloydb_database}",
            "user": "agentspace_user",
            "password": f"{agentspace_user_password}",
            "authMode": "AUTH_MODE_SERVICE_ACCOUNT"
          },
          "alloydb_ai_nl_config": { "nlConfigId": "default"}
        }
      }
}
params = {}

result = rest_api_helper(authed_session, url, 'POST', request_body, params)
result


### Create Schema Information

In [ ]:
url = f"https://discoveryengine.googleapis.com/v1beta/projects/{project_id}/locations/global/collections/default_collection/dataStores/{data_store_id}/schemas/default_schema"
request_body = {
  "structSchema": {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
      "products": { # table name
        "type": "object",
        "properties": { # column names
          "id": { "type": "bigint"},
          "cost": { "type": "bigint"},
          "category": { "type": "text"},
          "name": { "type": "text", "keyPropertyMapping": "title"}, # keyPropertyMapping = Use this column as Title
          "brand": { "type": "text"},
          "retail_price": { "type": "numeric"},
          "department": { "type": "text"},
          "sku": { "type": "text"},
          "distribution_center_id": { "type": "bigint"},
          "embedding": { "type": "vector"},
          "embedding_model_version": { "type": "text"},
          "product_description": { "type": "text", "keyPropertyMapping": "description"}, # keyPropertyMapping = Use this column as Description
          "product_description_embedding": { "type": "vector"},
          "product_description_embedding_model": { "type": "text"},
          "product_image_uri": { "type": "text", "keyPropertyMapping": "uri"}, # keyPropertyMapping = Use this column as URI
          "product_image_embedding": { "type": "vector"},
          "product_image_embedding_model": { "type": "text"},
          "fts_document": { "type": "txvector"},
          "sparse_embedding": { "type": "sparsevec"},
          "sparse_embedding_model": { "type": "text"}
        }
      }
    }
  }
}
params = {}

result = rest_api_helper(authed_session, url, 'PATCH', request_body, params)
result


In [ ]:
url = f"https://discoveryengine.googleapis.com/v1beta/projects/{project_id}/locations/global/collections/default_collection/dataStores/{data_store_id}/schemas/default_schema"
request_body = {
  "structSchema": {
    "$schema": "https://json-schema.org/draft/2020-12/schema",
    "type": "object",
    "properties": {
      "products": { # table name
        "type": "object",
        "properties": { # column names
          "id": { "type": "bigint"},
          "cost": { "type": "bigint"},
          "category": { "type": "text"},
          "name": { "type": "text", "keyPropertyMapping": "title"}, # keyPropertyMapping = Use this column as Title
          "brand": { "type": "text"},
          "retail_price": { "type": "numeric"},
          "department": { "type": "text"},
          "sku": { "type": "text"},
          "distribution_center_id": { "type": "bigint"},
          "embedding": { "type": "vector"},
          "embedding_model_version": { "type": "text"},
          "product_description": { "type": "text", "keyPropertyMapping": "description"}, # keyPropertyMapping = Use this column as Description
          "product_description_embedding": { "type": "vector"},
          "product_description_embedding_model": { "type": "text"},
          "product_image_uri": { "type": "text", "keyPropertyMapping": "uri"}, # keyPropertyMapping = Use this column as URI
          "product_image_embedding": { "type": "vector"},
          "product_image_embedding_model": { "type": "text"},
          "fts_document": { "type": "txvector"},
          "sparse_embedding": { "type": "sparsevec"},
          "sparse_embedding_model": { "type": "text"}
        }
      }
    }
  }
}
params = {}

result = rest_api_helper(authed_session, url, 'PATCH', request_body, params)
result


### Wait for Schema Operation

In [ ]:
import time
operation_id = result['name']

operation_complete = False
while operation_complete == False:
  print(f"Operation still running: {operation_id}")
  url = f"https://discoveryengine.googleapis.com/v1/{operation_id}"
  response = rest_api_helper(authed_session, url, 'GET', request_body, {})
  operation_complete = response['done']
  if operation_complete:
    print(f"Operation complete. Check result payload for potential errors. \nResult: {response}")
    continue
  time.sleep(5)

### Update UI Configuration

> NOTE: The [documentation](https://cloud.google.com/agentspace/agentspace-enterprise/docs/create-data-store#alloydb-ai-nl-setup) says this should be a PATCH, but [the API](https://cloud.google.com/generative-ai-app-builder/docs/reference/rest/v1alpha/projects.locations.collections.engines.widgetConfigs) doesn't support it yet. This step will have to be done manually until the API is updated to support PATCH.

In [ ]:
agentspace_id = "agentspace_1743448044898"
url = f"https://discoveryengine.googleapis.com/v1alpha/projects/{project_id}/locations/global/collections/default_collection/engines/{agentspace_id}/widgetConfigs/default_search_widget_config?updateMask=uiSettings"
request_body = {
  "uiSettings": {
    "dataStoreUiConfigs": [
      {
        "name": f"projects/{project_id}/locations/global/collections/default_collection/dataStores/{data_store_id}",
        "id": f"{data_store_id}",
        "fieldsUiComponentsMap": {
          "title": {
            "field": "title",
            "displayTemplate": "{value}"
          },
          "text1": {
            "field": "description",
            "displayTemplate": "{value}"
          },
          "url": {
            "field": "url",
            "displayTemplate": "{value}"
          }
        }
      }
    ],
    "interactionType": "SEARCH_ONLY"
  }
}
params = {}

result = rest_api_helper(authed_session, url, 'PATCH', request_body, params)
result


### Confirm the New Widget Config

In [ ]:
agentspace_id = "agentspace_1743448044898"
url = f"https://discoveryengine.googleapis.com/v1alpha/projects/{project_id}/locations/global/collections/default_collection/engines/{agentspace_id}/widgetConfigs/default_search_widget_config"
request_body = {}
params = {}

result = rest_api_helper(authed_session, url, 'GET', request_body, params)
result
